# Data Profiling Report (Week2) -- Yuchuan Cai

## Loading the Original Data

In [31]:
import pandas as pd

In [32]:
df = pd.read_csv("../data/processed/listing_sample.csv")
df

,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1137644469,556 D Street,Upland,3.0,3.0,535000,Stunning 3-Bedroom Condo in Upland Estates\r\r...
1,1151453101,1266 Wesley Avenue,Pasadena,4.0,2.0,1545000,Brigden Ranch Craftsman full of charm and warm...
2,1040333124,1181 Foxchase Drive,San Jose,2.0,1.0,669888,Coming Soon! Welcome home to this wonderful an...
3,1145639230,27875 Goetz Road,Menifee,5.0,4.0,1159000,Welcome to Your Dream Home in Menifee!\r\r\n\r...
4,1149603657,12500 Lull Street,North Hollywood,3.0,2.0,849999,Charming Corner Home with Endless Potential. T...
...,...,...,...,...,...,...,...
995,1148335355,2235 Stearns Road,Paradise,3.0,2.0,444500,Welcome to 2235 Stearns — a beautifully crafte...
996,1146844754,213 N Rancho Place,Pinole,3.0,3.0,875000,Discover the breathtaking beauty of Mt. Tamalp...
997,1151300741,1007 Woodward,Fresno,3.0,2.0,269000,Here is the home you have been looking for! Th...
998,1137580937,2468 Coldwater Canyon Drive,Beverly Hills,4.0,6.0,6395000,Step through the grand two-story entry with st...


## Reason for cleaning data -- sample data before profiling
Since the data in our original csv document contains a lot of useless information, we have to clean the data so as to move on in a better condition. This section will show how dirty the original data is

### Null Rates of the data

In [33]:
number_of_null = df['remarks'].isnull().sum()
null_rate = number_of_null / len(df['remarks'])
print(f"The null rate of the remarks in the data is: {null_rate}")

The null rate of the remarks in the data is: 0.0


Since we have cleaned the data by SQL command before we make it as a csv file, so the null rate == 0 totally make sense

### HTML Presence of the data

In [34]:
html_mask = df['remarks'].astype(str).str.contains(r'<[^>]+>', regex=True, case=False, na=False)
html_count = html_mask.sum()
html_rate = (html_count / len(df)) * 100
print(f"the rate of html presence in the remarks of the data is: {html_rate}")

the rate of html presence in the remarks of the data is: 0.0


This shows that the rate of html presence in this data base is also 0, which is a very good thing for us to move on. Although the html presence does not exsit in this data base, I still have the function to handle the html presence, in case that it might be useful in the future

### Common Abbreviations in the data
In this section, I will only show few samples of the abbreviations, since the total number of abbreviations might be very big

In [35]:
sqft_count = df['remarks'].astype(str).str.contains(r'\bsqft\b', regex=True, case=False, na=False).sum()
sf_count = df['remarks'].astype(str).str.contains(r'\bsf\b', regex=True, case=False, na=False).sum()
br_count = df['remarks'].astype(str).str.contains(r'\bbr\b', regex=True, case=False, na=False).sum()
ba_count = df['remarks'].astype(str).str.contains(r'\bba\b', regex=True, case=False, na=False).sum()

print(f"number of 'sqrt' {sqft_count}")
print(f"number of 'sf' {sf_count}")
print(f"number of 'br' {br_count}")
print(f"number of 'ba' {ba_count}")

number of 'sqrt' 27
number of 'sf' 16
number of 'br' 1
number of 'ba' 2


Since we only shown four samples of the abbreviations, and the total number of them have reached 46, which is a very large sample. At the same time, samples like 'sqrt' and 'sf' actually have same meaning, which is 'square feet', if we do not adjust this kind of difference, our training process will defintely be affected by them.

## Data After Profiling

Import our text_cleaner

In [36]:
import sys
sys.path.append('../scripts')
from text_cleaning import TextCleaner
cleaner = TextCleaner()

In [37]:
df["remarks_after_cleaning"] = df["remarks"].apply(cleaner.clean_text)
df.to_csv('cleaned_listing_sample.csv', index=False)
print(f"data after cleaning:\n{df["remarks_after_cleaning"]}")

data after cleaning:
0      stunning 3-bedroom condo in upland estates wel...
1      brigden ranch craftsman full of charm and warm...
2      coming soon! welcome home to this wonderful an...
3      welcome to your dream home in menifee! this st...
4      charming corner home with endless potential. t...
                             ...                        
995    welcome to 2235 stearns — a beautifully crafte...
996    discover the breathtaking beauty of mt. tamalp...
997    here is the home you have been looking for! th...
998    step through the grand two-story entry with st...
999    don't miss out on this absolutely gorgeous tow...
Name: remarks_after_cleaning, Length: 1000, dtype: str


Testing about the same abbreviations before data cleaning:

In [38]:
sqft_count = df['remarks_after_cleaning'].astype(str).str.contains(r'\bsqft\b', regex=True, case=False, na=False).sum()
sf_count = df['remarks_after_cleaning'].astype(str).str.contains(r'\bsf\b', regex=True, case=False, na=False).sum()
br_count = df['remarks_after_cleaning'].astype(str).str.contains(r'\bbr\b', regex=True, case=False, na=False).sum()
ba_count = df['remarks_after_cleaning'].astype(str).str.contains(r'\bba\b', regex=True, case=False, na=False).sum()

print(f"number of 'sqrt' after cleaning: {sqft_count}")
print(f"number of 'sf' after cleaning: {sf_count}")
print(f"number of 'br' after cleaning: {br_count}")
print(f"number of 'ba' after cleaning: {ba_count}")

number of 'sqrt' after cleaning: 0
number of 'sf' after cleaning: 0
number of 'br' after cleaning: 0
number of 'ba' after cleaning: 0


## Conclusion
As we can see, the data becomes much cleaner after we apply the TextCleaner